# Using Fragment Embeddings

## Estimated Time

- **CPU:** ~30 minutes
- **GPU:** ~30 minutes
- *Category: Medium*

## Overview

Load and use trained GAE embeddings as features for ML tasks.

## Prerequisites

- [x] GSGE installed (see [Installation Guide](../../docs/getting-started/installation.md))
- [ ] Trained GAE model with embeddings
- [ ] Basic understanding of PyTorch tensors

## Learning Objectives

By the end of this tutorial, you will be able to:
- Load trained embeddings from checkpoint
- Create embedding lookup tables
- Use embeddings for molecular property prediction

## Setup

Import the necessary modules:
- `torch`: PyTorch for tensor operations
- `GSGE`: Main GSGE class
- `GSGE_Embedding`: Embedding layer for using pre-trained fragment embeddings
- `get_tests_dir`: Utility to locate test fixtures

In [7]:
import torch
from GSGE import GSGE, GSGE_Embedding, get_tests_dir

print("[OK] Imports successful")

[OK] Imports successful


## Load Pre-trained GSGE Instance

Load a GSGE instance that contains trained fragment embeddings from a Graph Autoencoder (GAE).

In [8]:
# Load GSGE with saved fragment data
tests_dir = get_tests_dir()
if tests_dir is None:
    raise RuntimeError("Cannot find tests directory. Run from GSGE source checkout.")
pkl_path = tests_dir / 'test_gsge_save_with_descriptors.pkl'
gsge = GSGE(GSGE_load_path=pkl_path)

print(f"[OK] GSGE loaded successfully")
print(f"  - Pickle file: {pkl_path}")

Data loaded and restored into gsge from C:\Users\Admin\OneDrive - Universiteit Leiden\PhD\Repos\jasper\gsge\tests\test_gsge_save_with_descriptors.pkl
[OK] GSGE loaded successfully
  - Pickle file: C:\Users\Admin\OneDrive - Universiteit Leiden\PhD\Repos\jasper\gsge\tests\test_gsge_save_with_descriptors.pkl


## Get Fragment Embeddings

Extract the pre-trained fragment embeddings from the GSGE instance.

In [9]:
# Get the fragment embedding lookup table
look_up_table = gsge.get_fragment_embeddings()

if look_up_table is not None:
    print(f"[OK] Fragment embeddings loaded")
    print(f"  - Shape: {look_up_table.shape}")
    print(f"  - Number of fragments: {look_up_table.shape[0]}")
    print(f"  - Embedding dimension: {look_up_table.shape[1]}")
else:
    print("[!] No embeddings found. Train GAE first.")

[OK] Fragment embeddings loaded
  - Shape: torch.Size([185, 128])
  - Number of fragments: 185
  - Embedding dimension: 128


## Create Embedding Layer

Create a PyTorch embedding layer that uses the pre-trained fragment embeddings. The `only_token2vec=True` parameter means we're using only the pre-trained embeddings (no learned sparse embeddings).

In [10]:
# Get embedding size and vocabulary size
emb_size = look_up_table.shape[1]
token_vocab = gsge.get_GSGE_vocab()
vocab_size = len(token_vocab)
embedding_table_size = look_up_table.shape[0]  # Actual number of trained embeddings

print(f"Vocabulary size: {vocab_size}")
print(f"Embedding table size: {embedding_table_size}")
print(f"Embedding dimension: {emb_size}")
print(f"\nNote: Embedding table is smaller than vocabulary because:")
print("      - Some tokens (special tokens, single elements) may not have trained embeddings")
print("      - Only fragments present in GAE training have embeddings")

# Create embedding layer with pre-trained embeddings
# When only_token2vec=True, we use the pre-trained embeddings directly
embedding_layer = GSGE_Embedding(
    sparse_vocab_size=vocab_size,         # Total vocabulary size
    dense_size=emb_size,                   # Pre-trained embedding dimension
    embedding_dim=128,                     # For sparse tokens (not used when only_token2vec=True)
    GSGE_combined_embeddings=look_up_table, # Pre-trained embeddings
    only_token2vec=True,                   # Only use pre-trained embeddings
    no_grad=True                           # Freeze embeddings
)

print(f"\n[OK] Embedding layer created")
print(f"  - Sparse vocab size: {vocab_size}")
print(f"  - Dense embedding size: {emb_size}")
print(f"  - Weights frozen: {not any(p.requires_grad for p in embedding_layer.parameters())}")

Vocabulary size: 232
Embedding table size: 185
Embedding dimension: 128

Note: Embedding table is smaller than vocabulary because:
      - Some tokens (special tokens, single elements) may not have trained embeddings
      - Only fragments present in GAE training have embeddings

[OK] Embedding layer created
  - Sparse vocab size: 232
  - Dense embedding size: 128
  - Weights frozen: False


## Use the Embedding Layer

Create a random sequence of token IDs and pass them through the embedding layer to get the embedded representations.

In [11]:
# Create a random sequence of token IDs
# In practice, these would come from tokenized SMILES
batch_size = 4
seq_length = 10

# Use the actual embedding table size, not vocab size
# (some tokens like special tokens may not have trained embeddings)
embedding_table_size = look_up_table.shape[0]

random_sequence = torch.randint(
    low=0, 
    high=embedding_table_size,  # Use embedding table size, not vocab_size
    size=(batch_size, seq_length), 
    dtype=torch.long
)

print(f"Random token ID sequence:")
print(f"  - Shape: {random_sequence.shape}")
print(f"  - Values (first row): {random_sequence[0].tolist()}")
print(f"\nNote: Using embedding_table_size ({embedding_table_size}) not vocab_size ({vocab_size})")
print("      Some vocabulary tokens may not have trained embeddings.")

Random token ID sequence:
  - Shape: torch.Size([4, 10])
  - Values (first row): [157, 119, 60, 45, 36, 117, 58, 85, 119, 69]

Note: Using embedding_table_size (185) not vocab_size (232)
      Some vocabulary tokens may not have trained embeddings.


In [12]:
# Pass the sequence through the embedding layer
embedded_output = embedding_layer(random_sequence)

print(f"[OK] Embedding complete")
print(f"  - Input shape: {random_sequence.shape}")
print(f"  - Output shape: {embedded_output.shape}")
print(f"  - Output dtype: {embedded_output.dtype}")

[OK] Embedding complete
  - Input shape: torch.Size([4, 10])
  - Output shape: torch.Size([4, 10, 128])
  - Output dtype: torch.float32


In [13]:
# Examine the embedded output
print("Sample embedded output (first batch entry):")
print(f"  - First token embedding: {embedded_output[0, 0][:5].tolist()}...")  # First 5 values
print(f"  - All tokens for first sample: {embedded_output[0].shape}")

# Verify embeddings are frozen (no gradients)
print(f"\nEmbedding layer parameters:")
for name, param in embedding_layer.named_parameters():
    print(f"  - {name}: requires_grad={param.requires_grad}")

Sample embedded output (first batch entry):
  - First token embedding: [-0.5587894916534424, -1.1911150217056274, -1.4429839849472046, -0.7212543487548828, 1.064405083656311]...
  - All tokens for first sample: torch.Size([10, 128])

Embedding layer parameters:
  - sparse_embed.weight: requires_grad=True


## Using Embeddings with Real Tokenized Molecules

In practice, you would:
1. Tokenize SMILES using `gsge.preprocess_from_SMILES()`
2. Convert tokens to IDs using the vocabulary
3. Pass the token IDs through the embedding layer

### Example Workflow:

```python
# Tokenize a molecule
smiles = "CCO"
tokens = gsge.preprocess_from_SMILES(smiles)

# Convert to token IDs
token_ids = torch.tensor([[token_vocab[t] for t in tokens]])

# Get embeddings
embeddings = embedding_layer(token_ids)
```

### Next Steps:

- Use these embeddings as features in ML models
- Train sequence models (RNN, Transformer) on tokenized molecules
- Combine with other features (descriptors, graph features)